# Tool Schemas Claude Selects Correctly

*Definition, loop, and calling patterns.*

- Everything so far shaped **what Claude produces**. Tool use is different: you hand Claude a
  set of **actions** and trust it to pick the right one.
- That pick is driven almost entirely by **what you wrote in the schema**.

<details>
<summary><i>Full module text — intro</i></summary>

So far, the work has been about shaping what Claude produces: framing the request, giving
examples, picking the technique that fits the output you want. With tool-use, you're not
steering language toward a good answer anymore, you're handing Claude a set of actions and
trusting it to pick the right one; that pick is driven almost entirely by what you wrote in
the schema.

</details>

## The loop

⚠️ **The most common misconception: Claude does not run your tools.**

Claude reads your definitions, decides which fits, and *tells your application* what to call
and with which inputs. Your code executes it and sends the result back.

| # | Step | Who |
|---|---|---|
| 1 | Define schema — name, description, input_schema | You |
| 2 | Send message | You |
| 3 | `tool_use` block — which tool, what arguments | Claude |
| 4 | **Execute tool** | **You** |
| 5 | Return `tool_result` | You |
| 6 | Continue using the result | Claude |

- **The loop is not automatic.** Step 4 is yours. Miss it and Claude never gets the data it
  asked for — the loop breaks.
- The boundary between what Claude owns and what your code owns is **where most tool-use bugs
  live**.
- If the miss is *systematic* — wrong tool, every time — the fix is back at **step 1**.

<details>
<summary><i>Full module text — How the tool-use loop works</i></summary>

The most common misconception about tool-use is that Claude runs the tools. Instead, Claude
reads your tool definitions, decides which one fits the situation, and tells your application
what to call it along with the required inputs. Your application executes the tool, gets the
result, and sends it back; then Claude uses that result to continue.

This back-and-forth shouldn't be ignored in production: if your application does not handle
the return correctly, Claude never gets the data it asked for, and the loop breaks. The
boundary between what Claude owns and what your code owns is where most tool-use bugs live.

Define schema: You define a schema with a name, a description, and an input schema. Claude
reads this to decide whether and when to call the tool.

It's important to note that the loop is not automatic and you need to complete the fourth
step. If the miss is systematic, the fix is in the schema definition step.

</details>

In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()

model = "claude-opus-5"

## Steps 1–3: define a schema, send, receive a `tool_use` block

In [3]:
GET_BALANCE = {
    "name": "get_account_balance",          # specific beats generic: not "get_data"
    "description": (
        "Retrieve the current balance for a specific account ID. "
        "Returns the balance in USD as a number. "
        "Use this when the user asks how much money is in an account right now. "
        "Do not use this for transaction history or past payments."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "account_id": {
                "type": "string",
                "description": "Account identifier, e.g. 'ACC-1024'.",
            },
        },
        "required": ["account_id"],          # the call makes no sense without it
    },
}

messages = [{"role": "user", "content": "What's the balance on account ACC-1024?"}]

first = client.messages.create(
    model=model,
    max_tokens=1000,
    tools=[GET_BALANCE],
    messages=messages,
)

print("stop_reason:", first.stop_reason)      # 'tool_use' means Claude wants you to act
print("blocks:     ", [b.type for b in first.content])

for block in first.content:
    if block.type == "tool_use":
        print(f"\n  tool:  {block.name}")
        print(f"  id:    {block.id}")
        print(f"  input: {block.input}")

stop_reason: tool_use
blocks:      ['text', 'tool_use']

  tool:  get_account_balance
  id:    toolu_01Scnv2RE6RAoCEx4Cg3MEZL
  input: {'account_id': 'ACC-1024'}


## Steps 4–6: execute, return the result, let Claude continue

Two rules the code below obeys:

1. Append the assistant turn's **whole `content` array** — text blocks, thinking blocks and
   all. Rebuilding it from just the `tool_use` block corrupts the history.
2. Every `tool_result` carries the **same `tool_use_id`** as the call it answers, and lands
   in the **immediately following** user turn.

In [4]:
# --- your code owns this part ---
BALANCES = {"ACC-1024": 4210.55, "ACC-2048": 87.10}


def run_tool(name, args):
    """Execute a tool call and return (result_text, is_error)."""
    if name == "get_account_balance":
        account = args["account_id"]
        if account in BALANCES:
            return f"{BALANCES[account]:.2f} USD", False
        return f"No account named {account}", True      # <- surface failures, don't drop them
    return f"Unknown tool {name}", True


# 1. Append the assistant turn WHOLE.
messages.append({"role": "assistant", "content": first.content})

# 2. All tool_results for that turn go back in ONE user message, ids matching.
results = []
for block in first.content:
    if block.type == "tool_use":
        text, failed = run_tool(block.name, block.input)
        results.append({
            "type": "tool_result",
            "tool_use_id": block.id,        # must match exactly
            "content": text,
            "is_error": failed,             # optional; True when the tool failed
        })
        print(f"executed {block.name} -> {text!r} (is_error={failed})")

messages.append({"role": "user", "content": results})

# 6. Claude continues with the result in hand.
second = client.messages.create(
    model=model, max_tokens=1000, tools=[GET_BALANCE], messages=messages,
)
print("\nstop_reason:", second.stop_reason)
print(next(b.text for b in second.content if b.type == "text"))

executed get_account_balance -> '4210.55 USD' (is_error=False)

stop_reason: end_turn
Account ACC-1024 has a current balance of **$4,210.55 USD**.


## Message block structure

A tool-use conversation is **structured blocks, not plain text**. Four types do the work:

| Block | From | Contains | Critical rule |
|---|---|---|---|
| `text` | Claude | Prose output | Can arrive **alongside** a `tool_use` block in the same turn. Preserve the full `content` array when appending to history — dropping the text block corrupts context |
| `tool_use` | Claude | Tool name, unique **id**, input arguments | Must be answered by a `tool_result` with the same id, in the **immediately following** user turn |
| `tool_result` | You | Matching `tool_use_id`, result content, optional `is_error` | The id must match **exactly**. It's how Claude connects each result to its call — which matters when one turn issues several calls and results come back out of order |
| `thinking` | Claude (extended thinking only) | Internal reasoning + `signature` | Pass back **unchanged**. Any edit or summary breaks the signature. Redacted blocks too, encrypted or not |

⚠️ **The invariant:** every `tool_use` must have a matching `tool_result` in the *immediately
following* user turn. Missing, or deferred to a later turn → **API validation error**.

**This is structural. No amount of prompting fixes it** — your code has to emit the right
sequence on every request.

<details>
<summary><i>Full module text — Message block structure in a tool-use conversation</i></summary>

A tool-use conversation is built out of structured blocks, not plain text. Each assistant
turn and user turn is a list of blocks, and four block types do the work in a tool-use
session. A text block carries Claude's prose response. A tool_use block carries a tool call,
including the tool name, a unique ID, and the input arguments. A tool_result block carries
what your code returned after running the tool. A thinking block carries Claude's internal
reasoning, and it only appears when extended thinking is enabled.

The API enforces a specific pairing between these blocks. Every tool_use block in an
assistant turn must be answered by a tool_result block with a matching ID in the user turn
that immediately follows. If the IDs don't match, if the result is missing, or if the turns
are out of order, the request fails validation. This is not something you can fix by
adjusting your prompt; it's structural, and your code has to produce the sequence correctly
on every request.

The critical invariant is that every tool_use block from an assistant turn must have a
corresponding tool_result block in the immediately following user turn. Missing tool_result
blocks, or tool_result blocks that appear in a later turn rather than the immediately
following user turn, cause an API validation error.

</details>

### Break the pairing on purpose

Proof that this is enforced by the API, not by good manners:

In [5]:
import anthropic

# Same history, but the tool_result id is wrong.
bad_results = [dict(r, tool_use_id="toolu_deliberately_wrong") for r in results]
broken = [messages[0], {"role": "assistant", "content": first.content},
          {"role": "user", "content": bad_results}]

try:
    client.messages.create(model=model, max_tokens=1000, tools=[GET_BALANCE], messages=broken)
    print("no error raised")
except anthropic.BadRequestError as e:
    print(f"HTTP {e.status_code}")
    print(e.message)

HTTP 400
Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2.content.0: unexpected `tool_use_id` found in `tool_result` blocks: toolu_deliberately_wrong. Each `tool_result` block must have a corresponding `tool_use` block in the previous message.'}, 'request_id': 'req_011CenAkjfN2iDsrPKF61JZf'}


## Schema anatomy — what Claude actually reads

Three parts. **The description decides whether Claude selects the tool correctly.**

| Part | Guidance |
|---|---|
| **name** | Short and specific. `get_account_balance` ≫ `get_data` |
| **description** | The critical part. Write it in **two halves: when to use, and when not to** |
| **input_schema** | Parameters as JSON Schema. `required` only for what the call can't work without |

**Description, bad vs good:**

| | |
|---|---|
| ❌ | *"use this to find information"* — indistinguishable from any other retrieval tool |
| ✅ | *"use this to retrieve the current balance for a specific account ID **and do not use this for transaction history**"* — gives Claude an **exclusion condition** |

⚠️ **Overlapping parameter types between tools is the most common source of wrong-tool calls.**

<details>
<summary><i>Full module text — Schema anatomy</i></summary>

A tool schema has three parts, including name, description, and input_schema. The description
determines whether Claude selects the tool correctly or not.

Name: A short identifier that should be specific. For example, get_account_balance is more
useful to Claude than get_data.

Description: A critical part that Claude reads to decide whether a tool is required or not.
You should always write the description in two parts, including when to and when not to use
the tool:

- A description that says "use this to find information" will cause wrong selections because
  Claude cannot distinguish it from any other tool that retrieves something.
- A description that says "use this to retrieve the current balance for a specific account ID
  and do not use this for transaction history" gives Claude an exclusion condition to work
  with and is appropriately descriptive.

input_schema: Defines the parameters (the inputs your tool function accepts) using JSON
Schema. You should mark parameters as required when Claude requires them to call the tool
correctly. You can mark parameters as optional when the tool can operate without them.
Overlapping parameter types between tools is the most common source of wrong-tool calls.

</details>

## Five schema design decisions

| Decision | How to handle it | Why it matters |
|---|---|---|
| **Subtask dependency** | Output feeds the next call → must run **in sequence**. Independent → let Claude issue several `tool_use` blocks in one turn and run them concurrently | The one decision that changes schema design. Current models **default to parallel** when calls are independent. Model real dependencies as separate turns. `disable_parallel_tool_use` forces one call per turn |
| **Required fields** | Mark required **only** when the call makes no sense without it | Marking everything required forces Claude to **fabricate** values it has no basis to fill in |
| **Optional fields** | For parameters with sensible defaults, or where absence carries meaning. Leave out of `required`, give defaults in the function signature | Lets Claude omit what it doesn't have instead of guessing |
| **Description length** | **3–4 sentences**: what it does, when to reach for it, what it returns. Include input examples where format matters | Too short → Claude guesses. Too long → trigger conditions get buried under detail it doesn't reference at decision time |
| **Overlapping parameter types** | Add disambiguating language naming the **domain or trigger** each tool is for | Claude routes on **name + description**, parameter types secondary. Identical signatures → routing collapses to description alone |

<details>
<summary><i>Full module text — Decision table: Schema design choices</i></summary>

The schema is what Claude reads to decide which tool to call, what arguments to pass in, and
whether it has enough information to respond. A schema that's vague, under-described, or
missing required fields will produce tool calls that look syntactically correct but pick the
wrong tool, pass malformed inputs, or loop unnecessarily. The five decisions below determine
whether your implementation behaves predictably under real conditions. The table notes where
sequential and parallel tool-calling diverge.

**Subtask dependency** — When one tool's output feeds the next, the calls have to run in
sequence because the second call cannot be built until the first result comes back. When the
subtasks are independent of each other, you can structure the tool set so Claude issues
multiple tool_use blocks in a single turn and your code runs them concurrently. This is the
one decision that changes how you design the schema. Current Claude models default to
parallel calls when calls are independent. Where a real dependency exists, model it as
separate turns so the first result is available before the next call is built. Use
disable_parallel_tool_use to force one tool call per turn if needed.

**Required fields** — Mark a field as required only when the call doesn't make sense without
it. Place these in the required array of the input schema. Marking everything required forces
Claude to fabricate values for fields it has no basis to fill in. The required array is how
you tell Claude which inputs are non-negotiable.

**Optional fields** — Use optional fields for parameters with sensible defaults or where
absence carries meaning. Leave them out of the required array and give them defaults in the
function signature. Optional fields let Claude omit information it doesn't have, instead of
guessing. If a field is optional but marked required, every call must invent a value, which
can cause bad inputs.

**Description length** — Write three to four sentences per tool covering what it does, when
Claude should reach for it, and what it returns. Include examples of valid inputs where
format matters. If the description is too short, Claude guesses because there isn't enough
signal to distinguish your tool from others. If the description is too long, the trigger
conditions get buried under detail Claude doesn't reference at decision time.

**Overlapping parameter types** — When two tools accept the same parameter shape, add
disambiguating language to each description that names the domain or trigger the tool is
meant for. Claude routes on name plus description, with parameter types as a secondary
signal. When signatures are identical, routing collapses to description alone, and
similar-sounding descriptions become indistinguishable.

</details>

### Parallel calls: independent subtasks in one turn

In [6]:
GET_WEATHER = {
    "name": "get_weather",
    "description": (
        "Get the current temperature in Celsius for one city. "
        "Use this when the user asks about weather or temperature in a named place. "
        "Returns a single number. Call it once per city."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string", "description": "City name, e.g. 'Sydney'."}},
        "required": ["city"],
    },
}

parallel = client.messages.create(
    model=model,
    max_tokens=1000,
    tools=[GET_WEATHER],
    messages=[{"role": "user", "content": "Compare the temperature in Sydney and Melbourne."}],
)

calls = [b for b in parallel.content if b.type == "tool_use"]
print(f"tool_use blocks in ONE turn: {len(calls)}")
for b in calls:
    print(f"  {b.name}({b.input})")

tool_use blocks in ONE turn: 2
  get_weather({'city': 'Sydney'})
  get_weather({'city': 'Melbourne'})


In [7]:
# Force one call per turn — for when a real dependency exists between subtasks.
sequential = client.messages.create(
    model=model,
    max_tokens=1000,
    tools=[GET_WEATHER],
    tool_choice={"type": "auto", "disable_parallel_tool_use": True},
    messages=[{"role": "user", "content": "Compare the temperature in Sydney and Melbourne."}],
)

print("tool_use blocks:", len([b for b in sequential.content if b.type == "tool_use"]))

tool_use blocks: 1


## Worked example: descriptions that cause wrong-tool selection

> *Illustrative — names and descriptions constructed to demonstrate the disambiguation
> principle, not drawn from a production system.*

Two tools with **identical parameter shapes**. Distinct names, but both descriptions start
with *"use this to find information."* Claude routes on name **plus description**, with
parameter types only a secondary signal — so when signatures match and descriptions are
interchangeable, routing collapses to nothing.

In [7]:
import collections

SCHEMA = {"type": "object",
          "properties": {"query": {"type": "string"}},
          "required": ["query"]}

VAGUE = "Use this to find information."

SHARP_SEARCH = (
    "Use this to search the knowledge base when the user asks a question that requires "
    "looking up current information. Do not use this if the result of a prior search in "
    "this session already covers the question."
)
SHARP_CACHE = (
    "Use this to retrieve a result that was already fetched during this session. "
    "Only use this if search_knowledge_base was called earlier in this conversation "
    "for the same query."
)


def probe(search_desc, cache_desc, conversation, runs=5):
    """Run the same conversation N times; count which tool Claude picks."""
    tools = [
        {"name": "search_knowledge_base", "description": search_desc, "input_schema": SCHEMA},
        {"name": "get_cached_result", "description": cache_desc, "input_schema": SCHEMA},
    ]
    picked = []
    for _ in range(runs):
        r = client.messages.create(
            model=model, max_tokens=500, tools=tools,
            output_config={"effort": "low"}, messages=conversation,
        )
        names = [b.name for b in r.content if b.type == "tool_use"]
        picked.append(names[0] if names else "(no tool)")
    return dict(collections.Counter(picked))

### Scenario 1 — a fresh question

Nothing has been looked up yet, so only one tool is plausible. **Vague descriptions cost you
nothing here** — which is exactly why this failure hides during early testing.

In [9]:
QUESTION = "What is our refund window for annual plans?"
fresh = [{"role": "user", "content": QUESTION}]

print("vague:", probe(VAGUE, VAGUE, fresh))
print("sharp:", probe(SHARP_SEARCH, SHARP_CACHE, fresh))

vague: {'search_knowledge_base': 5}


sharp: {'search_knowledge_base': 5}


### Scenario 2 — the same query, already searched this session

Now **both tools are plausible** and the descriptions have to do real work. We build a
conversation where a search already ran and returned an answer, then ask again.

In [10]:
already_searched = [
    {"role": "user", "content": QUESTION},
    {"role": "assistant", "content": [
        {"type": "tool_use", "id": "toolu_prior1", "name": "search_knowledge_base",
         "input": {"query": "refund window annual plans"}},
    ]},
    {"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": "toolu_prior1",
         "content": "Annual plans have a 30-day refund window."},
    ]},
    {"role": "assistant", "content": "Annual plans have a 30-day refund window."},
    {"role": "user", "content": "Remind me again - what's the refund window for annual plans?"},
]

print("vague:", probe(VAGUE, VAGUE, already_searched))
print("sharp:", probe(SHARP_SEARCH, SHARP_CACHE, already_searched))

vague: {'(no tool)': 3, 'get_cached_result': 2}


sharp: {'get_cached_result': 5}


**Read the two scenarios together.** Scenario 1 looks fine either way. Scenario 2 is where
vague descriptions come apart — the routing becomes *inconsistent* across identical runs,
sometimes calling no tool at all. Sharp descriptions give the same answer every time.

That inconsistency is the real failure mode: not "always wrong", but **unreliable**, which is
harder to catch and worse in production.

⚠️ **Exclusion conditions depend on complete conversation history.** *"Only use this if
`search_knowledge_base` was called earlier"* is unevaluable if prior turns are truncated or
dropped — the logic then **fails silently**.

| | |
|---|---|
| **Handles well** | Routing to the right tool reliably, when descriptions are specific and exclusion conditions are stated |
| **Poor fit** | Two tools that do similar things and need ever-longer descriptions to keep apart → **merge them into one tool with a `type` parameter** |

Every extra tool you register increases the surface Claude reasons over. This discipline only
pays off when the underlying tools are genuinely distinct.

<details>
<summary><i>Full module text — Worked example</i></summary>

A developer registers two tools, including search_knowledge_base and get_cached_result. The
tool names are distinct, but Claude's tool selection weighs descriptions heavily; when
descriptions overlap, name alone is not sufficient to disambiguate. Both have descriptions
that start with "use this to find information." Without exclusion conditions, Claude
frequently selected the wrong tool on ambiguous inputs during development testing.

The problem is that both descriptions look identical to Claude at the point where the
selection decision is made. The fix is adding an additional sentence per description:

search_knowledge_base: "Use this to search the knowledge base when the user asks a question
that requires looking up current information. Do not use this if the result of a prior search
in this session already covers the question."

get_cached_result: "Use this to retrieve a result that was already fetched during this
session. Only use this if search_knowledge_base was called earlier in this conversation for
the same query."

The exclusion conditions give Claude a decision rule rather than two identical-looking
options. These conditions rely on complete conversation history being passed in each request.
If prior turns are truncated or dropped, Claude cannot evaluate them and the exclusion logic
silently fails.

Handles well: Routing Claude to the right tool reliably when descriptions are specific and
exclusion conditions are stated.

Poor fit: Two tools that do similar things and need ever-longer descriptions to keep apart:
at that point, merge them into one tool with a type parameter instead.

</details>

## MCP — when someone else has already written your tools

Everything above assumes **you** author the schema and the executing function. Often you
don't need to.

**Model Context Protocol (MCP)** is a standardised communication layer that moves tool
definitions and execution out of your application and into dedicated **servers**.

**Concrete case — GitHub.** Repos, PRs, issues, projects. Hand-rolling means a schema plus an
execution function for every piece, maintained as GitHub's API evolves. An MCP server has
already done that: you connect, receive the tool list, and Claude routes among them using the
**same description-based selection** you've been working with.

> **The mechanism is identical. What changes is who wrote the definitions and who owns them.**

**The loop does not change.** Claude still issues `tool_use`, you still execute and return
`tool_result`, the pairing rules still apply. Only **setup** differs: instead of registering
your own schemas, your MCP client sends a `ListToolsRequest`, gets the tool list, and passes
it to Claude. From Claude's side they're indistinguishable from hand-authored tools.

⚠️ **Context cost.** MCP servers add tool definitions to the context window **even when unused
this turn**. Connect several servers and the definitions consume budget before your first
message. Register only servers you're actively using, and check the cost against your window.

### Controlling load cost with `mcp_toolset`

Via the API MCP Connector, an `mcp_toolset` object in `tools` carries a `default_config`
applying to every tool on the server, overridable per tool through `configs` keyed by name.

| Setting | Effect |
|---|---|
| `defer_loading` | Delays loading a definition until the model needs it — cuts upfront context cost on servers with large tool lists |
| `enabled` | Turns individual tools on/off — register a server but expose only what you want seen |

⚠️ Requires the **`mcp-client-2025-11-20`** beta header. Without it the `mcp_toolset`
configuration does not apply as described.

Note this lives on the **beta** endpoint — `client.beta.messages.create`, not
`client.messages.create`.

In [11]:
# The request shape. Not executed: it needs a real MCP server URL.
# Note `mcp_servers` exists only on client.beta.messages.create — not the standard endpoint.

mcp_request = dict(
    model=model,
    max_tokens=1000,
    betas=["mcp-client-2025-11-20"],                 # required, or the config is ignored
    mcp_servers=[{
        "type": "url",
        "url": "https://example-mcp-server.invalid/mcp",   # placeholder
        "name": "github",
    }],
    tools=[{
        "type": "mcp_toolset",
        "mcp_server_name": "github",                 # must match the server name above
        "default_config": {"defer_loading": True},   # don't load every definition upfront
        "configs": {
            "create_pull_request": {"enabled": False},   # register the server, hide this tool
        },
    }],
    messages=[{"role": "user", "content": "List the open issues on my repo."}],
)

import json
print(json.dumps({k: v for k, v in mcp_request.items() if k != "messages"}, indent=2))

# response = client.beta.messages.create(**mcp_request)   # <- with a real server URL
print("\nboth halves are required: mcp_servers AND a matching mcp_toolset entry in tools")

{
  "model": "claude-opus-5",
  "max_tokens": 1000,
  "betas": [
    "mcp-client-2025-11-20"
  ],
  "mcp_servers": [
    {
      "type": "url",
      "url": "https://example-mcp-server.invalid/mcp",
      "name": "github"
    }
  ],
  "tools": [
    {
      "type": "mcp_toolset",
      "mcp_server_name": "github",
      "default_config": {
        "defer_loading": true
      },
      "configs": {
        "create_pull_request": {
          "enabled": false
        }
      }
    }
  ]
}

both halves are required: mcp_servers AND a matching mcp_toolset entry in tools


### Transports

| Server lives | Transport | How |
|---|---|---|
| **Local** | `stdio` | Your app spawns the server as a subprocess, talks over stdin/stdout |
| **Remote** | **Streamable HTTP** | Over the network — POST for client→server, optional GET-based SSE stream for server-initiated messages |

- An older **SSE-only transport is deprecated**. New integrations use Streamable HTTP.
- ⚠️ **Anthropic's API MCP Connector supports HTTP-exposed servers only.** stdio servers mean
  managing the MCP client connection yourself via the SDK.
- Once connected and definitions received, **your code treats both transports identically**.

## Choosing: MCP, manual, or both

| | |
|---|---|
| **Use MCP when** | A well-maintained server already exists for the service — check it covers the operations you need and tracks the current API. Writing those schemas yourself is overhead for no extra capability. ⚠️ The API MCP Connector supports **remote servers only**; local stdio servers need Claude Desktop or Claude Code as the client and cannot be connected through the API |
| **Write schemas manually when** | No server covers your case, or you need precise control over tool scope and description quality a general-purpose server won't give. Note the Connector *does* support allow/deny-listing per server via `MCPToolset` — so manual authoring is often justified by **description quality, not scope** |
| **Use both when** | Connect for **breadth**, then apply description-tuning to the specific tools you actually route to. Allowlist first to shrink the surface Claude reasons over, then sharpen descriptions. **Two separate levers — use both** |

<details>
<summary><i>Full module text — MCP</i></summary>

Everything in the previous sections assumes you are writing the tool schemas yourself: name,
description, input_schema, and the function that executes when Claude issues a tool_use
block. For many integrations, you do not need to do that. The Model Context Protocol, MCP, is
a standardized communication layer that moves tool definitions and execution out of your
application code and into dedicated servers. When an MCP server exists for the service you
want to reach, you can connect directly to the MCP server rather than building the
integration yourself.

Take a GitHub integration as a concrete case. GitHub exposes repositories, pull requests,
issues, projects, and more. To build a complete integration using the tool schema approach
from this module, you would need to write a schema and an execution function for every piece
of that functionality and maintain it as GitHub's API evolves. An MCP server for GitHub has
already done that. So, your application connects to the server, receives the full list of
available tools, and Claude selects among them using the same description-based routing you
have already been working with. The underlying mechanism is identical, but what changes is
who wrote it and who owns the tool definitions.

The loop you built earlier in this module does not change when you introduce MCP. Claude
still issues a tool_use block, your application still executes the tool and returns a
tool_result, and the message block pairing rules still apply. The difference is in the setup
step. Instead of registering schemas you wrote, your MCP client sends a ListToolsRequest to
the MCP server, receives the full tool list back, and passes those definitions to Claude.
From Claude's perspective, those tools are indistinguishable from ones you authored manually.

One practical implication worth noting: MCP servers add tool definitions to the context
window even when the tools are not being used in the current turn. If you connect several
servers at once, the tool definitions themselves consume budget before the first message
arrives. The schema design discipline from earlier in this module applies here too. Register
only the servers you are actively using, and check context cost against your window limit if
you are connecting multiple servers in the same session.

If you are using the API MCP Connector, you control loading cost through an mcp_toolset
object in the tools array. The mcp_toolset carries a default_config block that applies to
every tool on the server, and you can override individual tools through configs keyed by tool
name. Two settings matter for context cost:

- The defer_loading boolean, set inside default_config or a per-tool entry in configs, delays
  loading a tool definition until the model needs it, which reduces upfront context cost when
  you connect a server with a large tool list.
- The enabled boolean turns individual tools on or off, so you can register a server but
  expose only the tools you want the model to see. The MCP Connector requires the
  mcp-client-2025-11-20 beta header to be set on the request.

Without that header, the mcp_toolset configuration will not apply as described here.

The other piece worth knowing at this stage is how the client actually talks to the server.
MCP runs over one of two transports, and which one you use depends on where the server lives.
Local servers use stdio and your application spawns the server as a subprocess and
communicates over standard input and output. Remote servers use Streamable HTTP and your
application connects over the network via HTTP, using POST for client-to-server messages and
an optional GET-based SSE stream for server-initiated messages. An older SSE-only transport
exists but is deprecated, and new integrations should use Streamable HTTP. One constraint
worth flagging if you are using Anthropic's MCP connector in the API: only HTTP-exposed
servers are supported through the connector, and stdio servers require managing the MCP
client connection yourself via the SDK. Once the connection is established and tool
definitions are received, your application code treats both transports identically.

Use MCP when: A well-maintained MCP server already exists for the service you need (check
that it covers the specific operations you require and is actively maintained against the
service's current API. Writing and owning those schemas yourself adds implementation overhead
for no additional capability. Note that the Claude API MCP Connector only supports remote
servers. Local stdio servers require Claude Desktop or Claude Code as the client; they cannot
be connected directly through the API.

Write schemas manually when: No MCP server covers your use case, or when you need precise
control over tool scope and description quality that a general-purpose server does not
provide. Before defaulting to manual schemas for scope control, note that the API MCP
Connector supports allowlisting and denylisting specific tools per server via MCPToolset
configuration. Manual authoring may still be warranted for description quality, but not
always for scope.

Use both when: Connect to an MCP server for breadth then apply the description-tuning
discipline from earlier in this module to the specific tools you are actively routing to. MCP
and manual schema authoring are not mutually exclusive as the server gives you coverage, and
your descriptions give you precision where it matters. Apply tool allowlisting via MCPToolset
to limit the surface area Claude reasons over before layering in description tuning.
Narrowing the tool set and sharpening the descriptions are two separate levers, and you
should use both.

</details>

## Summary

| | |
|---|---|
| **Claude doesn't run tools** | It requests them. Step 4 — execution — is yours. Skip it and the loop breaks |
| **The invariant** | Every `tool_use` → a matching `tool_result` (same id) in the **immediately following** user turn. Structural, not promptable |
| **Preserve whole turns** | Append the full `content` array — text and thinking blocks included |
| **Description does the routing** | Name + description, parameters secondary. Write when to use **and when not to** |
| **`required` sparingly** | Everything-required makes Claude fabricate values |
| **Parallel by default** | Independent calls come back in one turn. `disable_parallel_tool_use` forces one |
| **MCP** | Same loop, someone else's schemas. Watch context cost; `defer_loading` + `enabled` control it. Beta header required; remote servers only through the API |